# 薛定谔方程 / Schrödinger equation

---

Time-dependent Schrödinger equation (nonrelativistic version) 
$$i \hbar \frac{d}{d t}|\Psi(t)\rangle=\hat{H}|\Psi(t)\rangle$$

Holding the Hamiltonian $\hat{H}$ constant, the Schrödinger equation has the solution,
$$|\Psi(t)\rangle=e^{-i \hat{H} t / \hbar}|\Psi(0)\rangle$$
The operater $\hat{U}=e^{-i \hat{H} t / \hbar}$ is known as the time-evolution operator.

Wave functions can form standing waves, called stationary states, described by Time-independent Schrödinger equation,
$$\hat{H}|\Psi\rangle=E\Psi$$
where Hamiltonian operator $\hat{H}=-\frac{\hbar^{2}}{2 m} \nabla^{2}+V(\mathbf{r})$ in the position representation.

<!-- bilingual -->

## Real-Space Grid
---

The simplest way to represent a real function $f(x)$, with $a ≤ x ≤ b$, is to sample it on a real-space grid of points $\left\{x_i\right\}$ from a to b with some uniform spacing h. The function is then represented by the vector of values $\left\{f (x_i)\right\}$

Second order linear differention in discrete space,
$$\frac{d^{2} f}{d x^{2}}=\lim _{h \rightarrow 0} \frac{f(x-h)-2 f(x)+f(x+h)}{h^{2}}$$  
[Laplace operator](https://en.wikipedia.org/wiki/Discrete_Laplace_operator) applied on discrete coordinates in [a, b]
$$
\frac{d^{2} f}{d x^{2}}=\begin{bmatrix}\begin{array}{ccccccc}-2 & 1 & 0 & \ldots & 0 & 0 \\ 1 & -2 & 1 & \ldots & 0 & 0 \\ 0 & 1 & -2 & \ldots & 0 & 0 \\ \ldots & \ldots & \ldots & \ldots & \ldots & \ldots \\ 0 & 0 & 0 & \ldots & -2 & 1 \\ 0 & 0 & 0 & \ldots & 1 & -2\end{array}\end{bmatrix}\begin{bmatrix}\begin{array}{c}f(a) \\ f(a+h) \\ f(a+2 h) \\ \ldots \\ f(a+(n-2) h) \\ f(b)\end{array}\end{bmatrix} / h^{2}=\operatorname{Lap}|f\rangle
$$

<!-- bilingual -->

In [ ]:
import scipy.sparse as sp

n_grid = 4
Lap_1D = sp.eye(n_grid, k=-1) + sp.eye(n_grid, k=1) - 2*sp.eye(n_grid)
Lap_2D = sp.kron(sp.eye(n_grid), Lap_1D) + sp.eye(n_grid**2, k=-n_grid) + sp.eye(n_grid**2, k=n_grid) - 2*sp.eye(n_grid**2)
Lap_3D = sp.kron(sp.eye(n_grid), Lap_2D) + sp.eye(n_grid**3, k=-n_grid**2) + sp.eye(n_grid**3, k=n_grid**2) - 2*sp.eye(n_grid**3)

## 束缚态和能级分立 / Bound states and discrete energy levels

---

1. 对于取值被“限制在一定空间范围内的”波函数、或者更准确说是在无穷远处收敛的波函数，它的能级是分立的，这样的状态叫做束缚态。
2. 对于无穷远处不收敛的波函数（例如自由粒子），能级则是连续的，这样的态叫做散射态。
3. 满足所谓周期性边界条件的波函数（例如氢原子的电子），它们在无穷远处不收敛，但能级也可能是分立的。

1. For wave functions whose values are "confined within a certain spatial range", or more precisely, converge at infinity, the energy levels are discrete; such states are called bound states.
2. For wave functions that do not converge at infinity (for example, free particles), the energy levels are continuous; such states are called scattering states.
3. For wave functions satisfying so-called periodic boundary conditions (for example, the electron of a hydrogen atom), they do not converge at infinity, but their energy levels can also be discrete.

<!-- bilingual -->

## 一维单粒子薛定谔方程 / One-dimensional single-particle Schrödinger equation

---

We will establish a matrix representation of the Hamiltonian of one partical $x \in [-5, 5]$ starting with the kinetic operator $-\frac{\hbar^{2}}{2 m}$ and a external potential $v_{ext}$.

<!-- bilingual -->

In [ ]:
%matplotlib inline
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg
import matplotlib.pyplot as plt
import time

class Schrodinger1D:
    ''' H|psi> = E |psi> '''
    def __init__(self, potential_func,
                 mass = 1, hbar = 1,
                 xmin=-5, xmax=5, ninterval=256):
        self.x = np.linspace(xmin, xmax, ninterval)
        self.Potential = sp.diags(potential_func(self.x), 0)
        self.Lap = self.laplacian((xmax - xmin)/ninterval, ninterval)
        self.Hamiltonian = - hbar**2 / (2*mass) * self.Lap + self.Potential
    def laplacian(self, dx, n_grid):
        return (sp.eye(n_grid, k=-1) + sp.eye(n_grid, k=1) - 2*sp.eye(n_grid)) / (dx**2)
    def eig_solve(self):
        time_start = time.time()
        eigValue, eigVector = sp.linalg.eigsh(self.Hamiltonian, k=16, which='SA')
        idx_sorted = np.argsort(eigValue)
        self.eigEnergy = eigValue[idx_sorted]
        self.eigVector = eigVector[:, idx_sorted]
        time_end = time.time()
        print("Times Used %.2f S"%(time_end - time_start))
    def plot_wavefunction(self, *args):
        fig, ax = plt.subplots(1, 1, figsize=(6, 5))
        for n in args:
            ax.plot(self.x, self.eigVector[:, n], label=r'$E_{%s}=%.2f$'%(n, self.eigEnergy[n]))
        ax.set_ylabel(r'$\psi(x)$')
        ax.set_xlabel(r'$x$')
        ax.legend()
    def plot_density(self, *args):
        fig, ax = plt.subplots(1, 1, figsize=(6, 5))
        for n in args:
            density = np.abs(self.eigVector[:, n]) ** 2
            ax.plot(self.x, density, label=r'$E_{%s}=%.2f$'%(n, self.eigEnergy[n]))
        ax.set_ylabel(r'$\rho(x)=\psi^*(x)\psi(x)$')
        ax.set_xlabel(r'$x$')
        ax.legend()

### 无限深方势井 / Infinite square well

---

<!-- bilingual -->

In [ ]:
def infinite_square_potential(x):
    return np.zeros_like(x)

schro_infinite_square = Schrodinger1D(infinite_square_potential)
schro_infinite_square.eig_solve()
print("Lowest Energies", schro_infinite_square.eigEnergy[0:5])

schro_infinite_square.plot_wavefunction(0, 1)

数值计算结果和解析解相符。

The numerical results agree with the analytical solution.

$$\phi_{n}=\sqrt{\frac{2}{L}}\sin{\frac{n\pi x}{L}}$$
$$E_{n}=\frac{n^2h^2}{8mL^2}$$

<!-- bilingual -->

粒子在简谐势阱中的分布概率密度

Probability density of the particle distribution in a harmonic potential well

<!-- bilingual -->

In [ ]:
schro_infinite_square.plot_density(0, 1)

### 简谐势井 / Harmonic potential well

---

<!-- bilingual -->

In [ ]:
def harmonic_potential(x, k=2):
    return 0.5 * k * x**2

schro_harmonic = Schrodinger1D(harmonic_potential)
schro_harmonic.eig_solve()
print("Lowest Energies", schro_harmonic.eigEnergy[0:5])

schro_harmonic.plot_wavefunction(0, 1)

粒子在简谐势阱中的分布概率密度

Probability density of the particle distribution in a harmonic potential well

<!-- bilingual -->

In [ ]:
schro_harmonic.plot_density(0, 1)

### 有限深方势井 / Finite square well

---

<!-- bilingual -->

In [ ]:
def square_well_potential(x, h=-20):
    u = np.zeros_like(x)
    u[abs(x) < 1] = h
    return u

schro_square_well = Schrodinger1D(square_well_potential)
schro_square_well.eig_solve()
print("Lowest Energies", schro_square_well.eigEnergy[0:5])

schro_square_well.plot_wavefunction(0, 1)

粒子在有限深方势阱中的分布概率密度

Probability density of the particle distribution in a finite square well

<!-- bilingual -->

In [ ]:
schro_square_well.plot_density(0, 1)

### 有限深双方势井 / Finite double square well

---

<!-- bilingual -->

In [ ]:
def double_well_potential(x, h=-20):
    u = np.zeros_like(x)
    u[(abs(x) > 0.5) & (abs(x) < 1)] = h
    return u

schro_double_well = Schrodinger1D(double_well_potential)
schro_double_well.eig_solve()
print("Lowest Energies", schro_double_well.eigEnergy[0:5])

schro_double_well.plot_wavefunction(0, 1)

粒子在有限深双势阱中的分布概率密度

Probability density of the particle distribution in a finite double well

<!-- bilingual -->

In [ ]:
schro_double_well.plot_density(0, 1)

可以看到，有限深双势井的基态是简并的，且粒子分布概率密度相同。

As we can see, the ground state of the finite double square well is degenerate, and the particle distribution probability densities are identical.

<!-- bilingual -->

### 量子叠加态和含时薛定谔方程 / Quantum superposition states and the time-dependent Schrödinger equation

---

Consider initial state to be a linear combination of groud state and 1st excitation state,
$$|\Psi(t=0)\rangle=(|\Psi_{E_{0}}\rangle + |\Psi_{E_{1}}\rangle)/\sqrt{2}$$
which is not stationary and its density distribution is time-dependent.
$$|\psi(t)\rangle=\frac{1}{\sqrt{2}}\left(\left|\psi_{E_{0}}\right\rangle \exp \left(\frac{-i E_{0} t}{\hbar}\right)+\left|\psi_{E_{1}}\right\rangle \exp \left(\frac{-i E_{1} t}{\hbar}\right)\right)$$
$$\rho(x, t)=\langle\psi(t)|\psi(t)\rangle=\frac{1}{2}\left[\left|\psi_{E_{0}}(x)\right|^{2}+\left|\psi_{E_{1}}(x)\right|^{2}+2\left|\psi_{E_{0}}(x)\right|\left|\psi_{E_{1}}(x)\right| \cos \left(\frac{(E_{0}-E_{1}) t}{\hbar}\right)\right]$$

<!-- bilingual -->

In [ ]:
def psit(t, hbar = 1):
    psi0 = schro_double_well.eigVector[:, 0]
    psi1 = schro_double_well.eigVector[:, 1]
    E0 = schro_double_well.eigEnergy[0]
    E1 = schro_double_well.eigEnergy[1]
    return 1/np.sqrt(2) * (psi0 * np.exp(-1j * E0 * t/hbar)
                        +  psi1 * np.exp(-1j * E1 * t/hbar))

density = np.abs(psit(0)) ** 2

fig, ax = plt.subplots(1, 1, figsize=(6, 5))
ax.plot(schro_double_well.x, density, label = r'$t = 0$')
ax.legend()

The partical is oscillating between the two wells with a period of $\frac{2\pi\hbar}{E_{0}-E_{1}}$

<!-- bilingual -->

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

class UpdateDist:
    def __init__(self, ax, x):
        self.success = 0
        self.line, = ax.plot([], [], 'k-')
        self.x = x
        self.ax = ax
        self.ax.set_xlim(-5, 5)
        self.ax.set_ylim(-0.01, 0.15)
        self.ax.grid(True)
    def __call__(self, i):
        time = i * 1
        psi = psit(t = time)
        density = np.abs(psi) ** 2
        self.line.set_data(self.x, density)
        self.line.set_label(r'$t=%s$'%(time))
        self.ax.legend()
        return self.line,

fig, ax = plt.subplots(1, 1, figsize=(5, 4), dpi=100)
ax.set_xlabel(r'$x$')
ud = UpdateDist(ax, x=schro_double_well.x)
ani = FuncAnimation(fig, ud, frames=130, interval=100, blit=True)
ani.save('/tmp/DoubleWell.gif', writer='pillow', fps=20)

In [ ]:
from IPython.display import Image
Image(filename ='DoubleWell.gif', width=500)

可以发现：

We can observe that:

1. 对于一维非奇性势能运动，粒子运动不简并，第n个激发态有n个零点。
2. 对于双无限深势井，第n和n+1个激发态能量简并。对应的物理图像是粒子无法穿越无限高势垒。

1. For motion in a one-dimensional non-singular potential, the particle motion is non-degenerate, and the n-th excited state has n zeros.
2. For a double infinite square well, the n-th and (n+1)-th excited state energies are degenerate. The corresponding physical picture is that the particle cannot cross an infinitely high potential barrier.

<!-- bilingual -->

### 基矢变换 / Basis transformation

---

选择一组基函数（基矢），将原函数表示为基函数的线性组合。原问题转变为求解基函数的组合系数。

Choose a set of basis functions (basis vectors) and express the original function as a linear combination of the basis functions. The original problem becomes that of solving for the combination coefficients of the basis functions.

$$\phi_{n}=\sqrt{\frac{2}{L}}\sin{\frac{n\pi x}{L}}$$
$$\phi_{old} = A \phi_{new}$$
$$H_{new} = A^T H_{old} A$$

<!-- bilingual -->

In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg
import matplotlib.pyplot as plt
import time

class Schrodinger1D_basis_set:
    ''' H|psi> = E |psi> '''
    def __init__(self, potential_func,
                 mass = 1, hbar = 1,
                 xmin=-5, xmax=5, ninterval=256):
        self.xmin = xmin
        self.xmax = xmax
        self.ninterval = ninterval
        self.x = np.linspace(xmin, xmax, ninterval)
        self.Potential = sp.diags(potential_func(self.x), 0)
        self.Lap = self.laplacian((xmax - xmin)/ninterval, ninterval)
        self.Hamiltonian = - hbar**2 / (2*mass) * self.Lap + self.Potential
    def laplacian(self, dx, n_grid):
        return (sp.eye(n_grid, k=-1) + sp.eye(n_grid, k=1) - 2*sp.eye(n_grid)) / (dx**2)       
    def basis_set(self, n):
        phi = np.zeros(self.ninterval)
        for i in range(self.ninterval):
            phi[i] = np.sqrt(2/self.ninterval) * np.sin((n+1)*np.pi*i/self.ninterval)
        return phi
    def eig_solve(self, nmax):
        ''' X_old = A X_new '''
        time_start = time.time()
        self.transition_matrix = np.zeros([self.ninterval, nmax])
        for n in range(nmax):
            self.transition_matrix[:,n] = self.basis_set(n)
        Hamiltonian_new = self.transition_matrix.T@self.Hamiltonian@self.transition_matrix
        '''Hamiltonian in the new basis sets is not sparse matrix'''
        eigValue, eigVector = np.linalg.eigh(Hamiltonian_new)  
        idx_sorted = np.argsort(eigValue)
        self.eigEnergy = eigValue[idx_sorted]
        self.eigVector = eigVector[:, idx_sorted]
        time_end = time.time()
        print("Times Used %.2f S"%(time_end - time_start))
    def plot_wavefunction(self, *args):
        fig, ax = plt.subplots(1, 1, figsize=(6, 5))
        for n in args:
            ax.plot(self.x, self.transition_matrix@self.eigVector[:, n], label=r'$E_{%s}=%.2f$'%(n, self.eigEnergy[n]))
        ax.set_ylabel(r'$\psi(x)$')
        ax.set_xlabel(r'$x$')
        ax.legend()
    def plot_density(self, *args):
        fig, ax = plt.subplots(1, 1, figsize=(6, 5))
        for n in args:
            density = np.abs(self.transition_matrix@self.eigVector[:, n]) ** 2
            ax.plot(self.x, density, label=r'$E_{%s}=%.2f$'%(n, self.eigEnergy[n]))
        ax.set_ylabel(r'$\rho(x)=\psi^*(x)\psi(x)$')
        ax.set_xlabel(r'$x$')
        ax.legend()

我们使用的基函数是无限深方势阱的能量本征态。对于无限深方势阱，基矢变换不会引入误差。

The basis functions we use are the energy eigenstates of the infinite square well. For the infinite square well, the basis transformation does not introduce any error.

<!-- bilingual -->

In [ ]:
def infinite_square_potential(x):
    return np.zeros_like(x)

schro_bs_infinite_square = Schrodinger1D_basis_set(infinite_square_potential)
schro_bs_infinite_square.eig_solve(8)  # 基矢数目为8
print("Energy", schro_bs_infinite_square.eigEnergy[:5])

schro_bs_infinite_square.plot_wavefunction(0, 1)

对于其他类型的势井，例如简谐势井，如果基矢组规模较小，拟合效果就会比较差。计算得到的本征值也会产生较大误差。

For other types of potential wells, such as the harmonic well, if the basis set is small, the fit will be poor. The computed eigenvalues will also have large errors.

<!-- bilingual -->

In [ ]:
def harmonic_potential(x, k=2):
    return 0.5 * k * x**2

schro_bs_harmonic = Schrodinger1D_basis_set(harmonic_potential)
schro_bs_harmonic.eig_solve(8)  # 基矢数目为8
print("Energy", schro_bs_harmonic.eigEnergy[0:5])

schro_bs_harmonic.plot_wavefunction(0, 1)

通过增加基矢数量，可以消除上述误差。

By increasing the number of basis functions, these errors can be eliminated.

<!-- bilingual -->

In [ ]:
schro_bs_harmonic.eig_solve(16)  # 基矢数目为16
print("Energy", schro_bs_harmonic.eigEnergy[0:5])

schro_bs_harmonic.plot_wavefunction(0, 1)

## 一维双粒子薛定谔方程 / One-dimensional two-particle Schrödinger equation

---

考虑一个无相互作用双粒子的波函数 $\phi(x_1, x_2)$，将其离散化到$N^2$个坐标格点$\left\{x_{1i}, x_{2j}\right\}$，对应的Hamiltonian矩阵的尺寸为$N^2 \times N^2$。

Consider the wave function $\phi(x_1, x_2)$ of two non-interacting particles, discretized onto $N^2$ grid points $\left\{x_{1i}, x_{2j}\right\}$. The size of the corresponding Hamiltonian matrix is $N^2 \times N^2$.

<!-- bilingual -->

In [ ]:
%matplotlib inline
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg
import matplotlib.pyplot as plt
import time

class Schrodinger2D:
    ''' H|psi> = E |psi> '''
    def __init__(self, potential_func,
                 mass = 1, hbar = 1,
                 xmin = -5, xmax = 5, ninterval=128):
        self.ninterval = ninterval
        self.x1 = np.linspace(xmin, xmax, ninterval)
        self.x2 = np.linspace(xmin, xmax, ninterval)
        potential = np.zeros(ninterval**2)
        for i in range(ninterval):
            for j in range(ninterval):
                potential[i*ninterval+j] = potential_func(self.x1[i], self.x2[j])
        self.Potential = sp.diags(potential)
        self.Lap = self.laplacian((xmax - xmin)/ninterval, ninterval)
        self.Hamiltonian = - hbar**2 / (2*mass) * self.Lap + self.Potential
    def laplacian(self, dx, n_grid):
        lap_1D = sp.eye(n_grid, k=-1) + sp.eye(n_grid, k=1) - 2 * sp.eye(n_grid)
        lap_2D = sp.kron(sp.eye(n_grid), lap_1D) + sp.eye(n_grid**2, k=-n_grid) + sp.eye(n_grid**2, k=n_grid) - 2*sp.eye(n_grid**2)
        return lap_2D / (dx**2)
    def eig_solve(self):
        time_start = time.time()
        eigValue, eigVector = sp.linalg.eigsh(self.Hamiltonian, k=16, which='SA')
        idx_sorted = np.argsort(eigValue)
        self.eigEnergy = eigValue[idx_sorted]
        self.eigVector = eigVector[:, idx_sorted]
        time_end = time.time()
        print("Times Used %.2f S"%(time_end - time_start))
    def plot_wavefunction(self, n):
        wave_func = np.reshape(self.eigVector[:, n], (self.ninterval, self.ninterval))
        fig, ax = plt.subplots(1, 1, figsize=(6, 5))
        p = ax.imshow(wave_func.real, cmap='hot')
        cb = fig.colorbar(p, shrink=0.8)
        cb.set_label(r"$\phi(x_1, x_2).real$", fontsize=18)
        ax.set_xlabel(r'$x_1$')
        ax.set_ylabel(r'$x_2$')
        ax.set_title(r'$E_{%s}=%.4f$'%(n, self.eigEnergy[n]), loc='left', fontsize=16)
    def plot_density(self, *args):
        fig, ax = plt.subplots(1, 1, figsize=(6, 5))
        for n in args:
            density = np.zeros(self.ninterval)
            for i in range(self.ninterval):
                for j in range(self.ninterval):
                    density[i] += np.abs(self.eigVector[:, n][i*self.ninterval+j]) ** 2
                    density[j] += np.abs(self.eigVector[:, n][i*self.ninterval+j]) ** 2
            ax.plot(density, label=r'$E_{%s}=%.4f$'%(n, self.eigEnergy[n]))
        ax.set_xlabel(r'$x$')
        ax.set_ylabel(r'$\rho=\psi^*\psi$')
        ax.legend()

### 无限深方势井 / Infinite square well

---

<!-- bilingual -->

In [ ]:
def infinite_square_potential(x1, x2):
    return 0.0

schro_infinite_square = Schrodinger2D(infinite_square_potential)
schro_infinite_square.eig_solve()

print("Lowest Energies", schro_infinite_square.eigEnergy[:6])

我们可以将双粒子波函数的解写为：
$$\Phi=C_{1}\phi_{n}(x_{1})\phi_{m}(x_{2}) + C_{2}\phi_{m}(x_{1})\phi_{n}(x_{2})$$
其中$\phi$为单粒子波函数。

We can write the solution of the two-particle wave function as:
$$\Phi=C_{1}\phi_{n}(x_{1})\phi_{m}(x_{2}) + C_{2}\phi_{m}(x_{1})\phi_{n}(x_{2})$$
where $\phi$ is a single-particle wave function.

对应的能量为：
$$E=(n^2+m^2)\frac{\pi\hbar^2}{2ma^2}$$

The corresponding energy is:
$$E=(n^2+m^2)\frac{\pi\hbar^2}{2ma^2}$$

<!-- bilingual -->

In [ ]:
schro_infinite_square.plot_wavefunction(0)

In [ ]:
schro_infinite_square.plot_density(0, 1, 2)

根据量子力学的全同粒子假设，多粒子波函数的坐标交换之后，新状态应当与原状态是不可区分的。

According to the identical-particle postulate of quantum mechanics, after exchanging the coordinates of a multi-particle wave function, the new state should be indistinguishable from the original state.

$$\hat{P_{ij}} \Phi = C \Phi$$

其中$C=1$（交换对称）或者$C=-1$（交换反对称）。

where $C=1$ (exchange symmetric) or $C=-1$ (exchange antisymmetric).

我们下面将检查数值求解得到的波函数的对称性：

Next we will check the symmetry of the numerically obtained wave function:

<!-- bilingual -->

In [ ]:
def check_symmetry(wave_func):
    wave_func_trans = np.transpose(wave_func)
    vector_exchange = wave_func_trans.flatten()
    wave_sum = (wave_func + wave_func_trans)
    wave_diff = (wave_func - wave_func_trans)
    if np.max(abs(wave_sum)) < 1e-02:
        print("Exchange Antisymmetry %.2f" %(np.max(abs(wave_sum))))
    elif np.max(abs(wave_diff)) < 1e-02:
        print("Exchange Symmetry %.2f" %(np.max(abs(wave_diff))))
    else:
        print("No Exchange Symmetry %.2f %.2f" %(np.max(abs(wave_sum)), np.max(abs(wave_diff))))
for n in range(8):
    wave_func = np.reshape(schro_infinite_square.eigVector[:, n], (schro_infinite_square.ninterval, schro_infinite_square.ninterval))
    print("Energy %.4f Orbital %.2d  " %(np.real(schro_infinite_square.eigEnergy[n]), n+1), end="")
    check_symmetry(wave_func)

波色子交换对称，
$$\Phi_{n,m}=\frac{1}{\sqrt{2}}(\phi_{n}(x_{1})\phi_{m}(x_{2})+\phi_{n}(x_{2})\phi_{m}(x_{1}))$$

Bosons are exchange symmetric,
$$\Phi_{n,m}=\frac{1}{\sqrt{2}}(\phi_{n}(x_{1})\phi_{m}(x_{2})+\phi_{n}(x_{2})\phi_{m}(x_{1}))$$

费米子交换反对称，
$$\Phi_{n,m}=\frac{1}{\sqrt{2}}(\phi_{n}(x_{1})\phi_{m}(x_{2})-\phi_{n}(x_{2})\phi_{m}(x_{1})), n \neq m$$

Fermions are exchange antisymmetric,
$$\Phi_{n,m}=\frac{1}{\sqrt{2}}(\phi_{n}(x_{1})\phi_{m}(x_{2})-\phi_{n}(x_{2})\phi_{m}(x_{1})), n \neq m$$

<!-- bilingual -->

### 库伦相互作用 / Coulomb interaction

考虑两个带电粒子，例如电子（我们的方程中没有考虑自旋）。

Consider two charged particles, for example electrons (spin is not considered in our equation).

<!-- bilingual -->

In [ ]:
def coulomb_potential(x1, x2, k=1):
    return k / (abs(x1-x2) + 1e-6)

schro_coulomb = Schrodinger2D(coulomb_potential)
schro_coulomb.eig_solve()

print("Energy", schro_coulomb.eigEnergy[:6])

In [ ]:
schro_coulomb.plot_wavefunction(0)

In [ ]:
schro_coulomb.plot_wavefunction(1)

In [ ]:
schro_coulomb.plot_density(0, 1, 2, 3)

我们再来检查轨道的对称性：

Let us again check the symmetry of the orbitals:

<!-- bilingual -->

In [ ]:
for n in range(8):
    wave_func = np.reshape(schro_coulomb.eigVector[:, n], (schro_coulomb.ninterval, schro_coulomb.ninterval))
    print("Energy %.4f Orbital %.2d  " %(np.real(schro_coulomb.eigEnergy[n]), n+1), end="")
    check_symmetry(wave_func)

可以发现，库伦排斥下，能量为二重简并，且简并的两个轨道分别为交换对称和交换反对称。

We can see that under Coulomb repulsion, the energy is doubly degenerate, and the two degenerate orbitals are exchange-symmetric and exchange-antisymmetric respectively.

<!-- bilingual -->

### 包含库伦相互作用的简谐势井 / Harmonic well including Coulomb interaction

---

<!-- bilingual -->

In [ ]:
def harmonic_coulomb(x1, x2, k_h=2, k_c=1):
    return 0.5 * k_h * (x1**2 + x2**2) + k_c / (abs(x1-x2) + 1e-6)

schro_harmonic_coulomb = Schrodinger2D(harmonic_coulomb)
schro_harmonic_coulomb.eig_solve()

print("Energy", np.real(schro_harmonic_coulomb.eigEnergy[:8]))

In [ ]:
schro_harmonic_coulomb.plot_wavefunction(0)

In [ ]:
schro_harmonic_coulomb.plot_density(0, 1, 2, 3)

### 基矢变换 / Basis transformation

我们选择方势井波函数的本征态作为新基矢，对待求解波函数进行展开：

We choose the eigenstates of the square-well wave functions as the new basis and expand the target wave function:

$$\phi_{n,m}=\frac{2}{a}\sin{\frac{n\pi x_{1}}{a}}\sin{\frac{m\pi x_{2}}{a}}$$

$$\phi_{old} = A \phi_{new}$$

相应的，Hamiltonian矩阵转换为：
$$H_{new} = A^T H_{old} A$$

Correspondingly, the Hamiltonian matrix is transformed to:
$$H_{new} = A^T H_{old} A$$

<!-- bilingual -->

In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg
import matplotlib.pyplot as plt
import time

class Schrodinger2D_basis_set:
    ''' H|psi> = E |psi> '''
    def __init__(self, potential_func,
                 mass = 1, hbar = 1,
                 xmin = -5, xmax = 5, ninterval=128):
        self.ninterval = ninterval
        self.x1 = np.linspace(xmin, xmax, ninterval)
        self.x2 = np.linspace(xmin, xmax, ninterval)
        potential = np.zeros(ninterval**2)
        for i in range(ninterval):
            for j in range(ninterval):
                potential[i*ninterval+j] = potential_func(self.x1[i], self.x2[j])
        self.Potential = sp.diags(potential)
        self.Lap = self.laplacian((xmax - xmin)/ninterval, ninterval)
        self.Hamiltonian = - hbar**2 / (2*mass) * self.Lap + self.Potential
    def laplacian(self, dx, n_grid):
        lap_1D = sp.eye(n_grid, k=-1) + sp.eye(n_grid, k=1) - 2 * sp.eye(n_grid)
        lap_2D = sp.kron(sp.eye(n_grid), lap_1D) + sp.eye(n_grid**2, k=-n_grid) + sp.eye(n_grid**2, k=n_grid) - 2*sp.eye(n_grid**2)
        return lap_2D / (dx**2)
    def basis_set(self, ninterval, n, m):
        phi = np.zeros(ninterval*ninterval)
        for j in range(ninterval):
            for i in range(ninterval):
                phi[j*ninterval+i] = 2/ninterval * np.sin((n+1)*np.pi*i/ninterval) * np.sin((m+1)*np.pi*j/ninterval)
        return phi
    def eig_solve(self, nmax):
        ''' X_old = A X_new '''
        time_start = time.time()
        self.transition_matrix = np.zeros([self.ninterval**2, nmax**2])
        for m in range(nmax):
            for n in range(nmax):
                self.transition_matrix[:,m*nmax + n] = self.basis_set(self.ninterval, n, m)
        Hamiltonian_new = self.transition_matrix.T@self.Hamiltonian@self.transition_matrix
        '''Hamiltonian in the new basis sets is not sparse matrix'''
        eigValue, eigVector = np.linalg.eigh(Hamiltonian_new)
        idx_sorted = np.argsort(eigValue)
        self.eigEnergy = eigValue[idx_sorted]
        self.eigVector = eigVector[:, idx_sorted]
        time_end = time.time()
        print("Times Used %.2f S"%(time_end - time_start))
    def plot_wavefunction(self, n):
        wave_func = np.reshape(self.transition_matrix@self.eigVector[:, n], (self.ninterval, self.ninterval))
        fig, ax = plt.subplots(1, 1, figsize=(6, 5))
        p = ax.imshow(wave_func.real, cmap='hot')
        cb = fig.colorbar(p, shrink=0.8)
        cb.set_label(r"$\phi(x_1, x_2).real$", fontsize=18)
        ax.set_xlabel(r'$x_1$')
        ax.set_ylabel(r'$x_2$')
        ax.set_title(r'$E_{%s}=%.4f$'%(n, self.eigEnergy[n]), loc='left', fontsize=16)
    def plot_density(self, *args):
        fig, ax = plt.subplots(1, 1, figsize=(6, 5))
        for n in args:
            density = np.zeros(self.ninterval)
            density_2D = np.abs(self.transition_matrix@self.eigVector[:, n])** 2
            for i in range(self.ninterval):
                for j in range(self.ninterval):
                    density[i] += density_2D[i*self.ninterval+j]
                    density[j] += density_2D[i*self.ninterval+j]
            ax.plot(density, label=r'$E_{%s}=%.4f$'%(n, self.eigEnergy[n]))
        ax.set_xlabel(r'$x$')
        ax.set_ylabel(r'$\rho=\psi^*\psi$')
        ax.legend()

In [ ]:
schro_bs_coulomb = Schrodinger2D_basis_set(coulomb_potential)
schro_bs_coulomb.eig_solve(8)  # 基矢数目为 8^2=64

print("Energy", np.real(schro_bs_coulomb.eigEnergy[:8]))

将待求解波函数使用合适的基矢组展开有可能降低计算量和加快计算速度，同时所得结果的精度也是可以接受的。

Expanding the target wave function in an appropriate basis set can potentially reduce the computational cost and speed up computation, while keeping the accuracy of the result acceptable.

<!-- bilingual -->

In [ ]:
schro_bs_coulomb.plot_wavefunction(0)

In [ ]:
schro_bs_coulomb.plot_density(0, 1, 2, 3)

## 三维单粒子薛定谔方程 / Three-dimensional single-particle Schrödinger equation

---

<!-- bilingual -->

In [ ]:
%matplotlib inline
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg
import matplotlib.pyplot as plt
from skimage import measure
import time

class Schrodinger3D:
    ''' H|psi> = E |psi> '''
    def __init__(self, potential_func,
                 mass = 1, hbar = 1,
                 xmin=-5, xmax=5, ymin=-5, ymax=5, zmin=-5, zmax=5,
                 ninterval=32):
        self.ninterval = ninterval
        self.x = np.linspace(xmin, xmax, ninterval)
        self.y = np.linspace(ymin, ymax, ninterval)
        self.z = np.linspace(zmin, zmax, ninterval)
        potential = np.zeros(ninterval**3)
        for i in range(ninterval):
            for j in range(ninterval):
                for k in range(ninterval):
                    potential[i*ninterval**2+j*ninterval+k] = potential_func(self.x[i], self.y[j], self.y[k])
        self.Potential = sp.diags(potential)
        self.Lap = self.laplacian((xmax - xmin)/ninterval, (ymax - ymin)/ninterval, (zmax - zmin)/ninterval, ninterval)
        self.Hamiltonian = - hbar**2 / (2*mass) * self.Lap + self.Potential
    def laplacian(self, dx, dy, dz, n_grid):
        lap_1D = (sp.eye(n_grid, k=-1) + sp.eye(n_grid, k=1) - 2 * sp.eye(n_grid)) / dx**2
        lap_2D = sp.kron(sp.eye(n_grid), lap_1D) + (
                 sp.eye(n_grid**2, k=-n_grid) + sp.eye(n_grid**2, k=n_grid) - 2*sp.eye(n_grid**2)) / dy**2
        lap_3D = sp.kron(sp.eye(n_grid), lap_2D) + (
                 sp.eye(n_grid**3, k=-n_grid**2) + sp.eye(n_grid**3, k=n_grid**2) - 2*sp.eye(n_grid**3)) / dz**2
        return lap_3D
    def eig_solve(self):
        time_start = time.time()
        eigValue, eigVector = sp.linalg.eigsh(self.Hamiltonian, k=16, which='SA')
        idx_sorted = np.argsort(eigValue)
        self.eigEnergy = eigValue[idx_sorted]
        self.eigVector = eigVector[:, idx_sorted]
        time_end = time.time()
        print("Times Used %.2f S"%(time_end - time_start))
    def plot_density_2D(self, n, slice = 8):
        fig, ax = plt.subplots(1, 1, figsize=(6, 5))
        wave_func = np.reshape(self.eigVector[:, n], (self.ninterval, self.ninterval, self.ninterval))
        density = np.abs(wave_func) ** 2
        ax.imshow(density[:,:,slice], cmap='hot')
    def plot_density_3D(self, n, iso_val = 0.0002):
        fig, ax = plt.subplots(1, 1, figsize=(6, 5), subplot_kw={'projection': '3d'})
        wave_func = np.reshape(self.eigVector[:, n], (self.ninterval, self.ninterval, self.ninterval))
        density = abs(wave_func) ** 2
        verts, faces, _, _ = measure.marching_cubes(density, iso_val)
        ax.plot_trisurf(verts[:, 0], verts[:,1], faces, verts[:, 2],
                        cmap='Spectral', lw=1)

### 无限深方势井 / Infinite square well

---

<!-- bilingual -->

In [ ]:
def infinite_square_potential(x, y, z):
    return 0.0

schro_infinite_square = Schrodinger3D(infinite_square_potential)
schro_infinite_square.eig_solve()

print("Lowest Energies", schro_infinite_square.eigEnergy[0:12])

In [ ]:
schro_infinite_square.plot_density_2D(0, slice=16)

In [ ]:
#%matplotlib widget
schro_infinite_square.plot_density_3D(0)

### 简谐势井 / Harmonic potential well

---

<!-- bilingual -->

In [ ]:
def harmonic_potential(x, y, z, k = 10):
    return 0.5 * k * (x**2 + y**2 + z**2)

schro_harmonic = Schrodinger3D(harmonic_potential)
schro_harmonic.eig_solve()

print("Lowest Energies", schro_harmonic.eigEnergy[0:12])

能量$\frac{1}{2}n(n+1)$度简并

Energy is $\frac{1}{2}n(n+1)$-fold degenerate

<!-- bilingual -->

### 库伦势井 / Coulomb potential well

---

氢原子模型

Hydrogen atom model

<!-- bilingual -->

In [ ]:
def coulomb_potential(x, y, z, k = 4):
    return -k / (np.sqrt(x**2 + y**2 + z**2) + 1e-6)

schro_coulomb = Schrodinger3D(coulomb_potential)
schro_coulomb.eig_solve()

print("Lowest Energies", schro_coulomb.eigEnergy[0:12])

能量$n^2$度简并

Energy is $n^2$-fold degenerate

<!-- bilingual -->

#### 氢原子的电子轨道 / Electron orbitals of the hydrogen atom

**1s轨道**

**1s orbital**

<!-- bilingual -->

In [ ]:
schro_coulomb.plot_density_3D(0)

**2p轨道**

**2p orbital**

<!-- bilingual -->

In [ ]:
schro_coulomb.plot_density_3D(1)

**3d轨道**

**3d orbital**

<!-- bilingual -->

In [ ]:
schro_coulomb.plot_density_3D(9)

### 基矢变换 / Basis transformation

我们选择方势井波函数的本征态作为新基矢，对待求解波函数进行展开：
$$\phi_{n,m,p}=\sqrt{\frac{8}{L_{x}L_{y}L_{z}}}\sin{\frac{n\pi x}{L_{x}}}\sin{\frac{m\pi y}{L_{y}}}\sin{\frac{p\pi z}{L_{z}}}$$

We choose the eigenstates of the square-well wave functions as the new basis and expand the target wave function:
$$\phi_{n,m,p}=\sqrt{\frac{8}{L_{x}L_{y}L_{z}}}\sin{\frac{n\pi x}{L_{x}}}\sin{\frac{m\pi y}{L_{y}}}\sin{\frac{p\pi z}{L_{z}}}$$

$$\phi_{old} = A \phi_{new}$$

相应的，Hamiltonian矩阵转换为：
$$H_{new} = A^T H_{old} A$$

Correspondingly, the Hamiltonian matrix is transformed to:
$$H_{new} = A^T H_{old} A$$

<!-- bilingual -->

In [ ]:
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
from skimage import measure
import time

class Schrodinger3D_basis_set:
    ''' H|psi> = E |psi> '''
    def __init__(self, potential_func,
                 mass = 1, hbar=1,
                 xmin=-5, xmax=5, ymin=-5, ymax=5, zmin=-5, zmax=5,
                 ninterval=32):
        self.ninterval = ninterval
        self.x = np.linspace(xmin, xmax, ninterval)
        self.y = np.linspace(ymin, ymax, ninterval)
        self.z = np.linspace(zmin, zmax, ninterval)
        potential = np.zeros(ninterval**3)
        for i in range(ninterval):
            for j in range(ninterval):
                for k in range(ninterval):
                    potential[i*ninterval**2+j*ninterval+k] = potential_func(self.x[i], self.y[j], self.y[k])
        self.Potential = sp.diags(potential)
        self.Lap = self.laplacian((xmax - xmin)/ninterval, (ymax - ymin)/ninterval, (zmax - zmin)/ninterval, ninterval)
        self.Hamiltonian = - hbar**2 / (2*mass) * self.Lap + self.Potential
    def laplacian(self, dx, dy, dz, n_grid):
        lap_1D = (sp.eye(n_grid, k=-1) + sp.eye(n_grid, k=1) - 2 * sp.eye(n_grid)) / dx**2
        lap_2D = sp.kron(sp.eye(n_grid), lap_1D) + (
                 sp.eye(n_grid**2, k=-n_grid) + sp.eye(n_grid**2, k=n_grid) - 2*sp.eye(n_grid**2)) / dy**2
        lap_3D = sp.kron(sp.eye(n_grid), lap_2D) + (
                 sp.eye(n_grid**3, k=-n_grid**2) + sp.eye(n_grid**3, k=n_grid**2) - 2*sp.eye(n_grid**3)) / dz**2
        return lap_3D
    def basis_set(self, ninterval, N, M, P):
        phi = np.zeros(ninterval**3)
        for k in range(ninterval):
            for j in range(ninterval):
                for i in range(ninterval):
                    phi[k*ninterval**2+j*ninterval+i] = ((2/ninterval)**1.5 * np.sin((N+1)*np.pi*i/ninterval) *
                                                        np.sin((M+1)*np.pi*j/ninterval) * np.sin((P+1)*np.pi*k/ninterval))
        return phi
    def eig_solve(self, nmax):
        time_start = time.time()
        self.transition_matrix = np.zeros([self.ninterval**3, nmax**3])
        for p in range(nmax):
            for m in range(nmax):
                for n in range(nmax):
                    self.transition_matrix[:,p*nmax**2 + m*nmax + n] = self.basis_set(self.ninterval, n, m, p)
        Hamiltonian_new = self.transition_matrix.T@self.Hamiltonian@self.transition_matrix
        eigValue, eigVector = np.linalg.eigh(Hamiltonian_new)         
        idx_sorted = np.argsort(eigValue)
        self.eigEnergy = eigValue[idx_sorted]
        self.eigVector = eigVector[:, idx_sorted]
        time_end = time.time()
        print("Times Used %.2f S"%(time_end - time_start))
    def plot_density_2D(self, n, slice = 8):
        fig, ax = plt.subplots(1, 1, figsize=(6, 5))
        wave_func = np.reshape(self.transition_matrix@self.eigVector[:, n], (self.ninterval, self.ninterval, self.ninterval))
        density = abs(wave_func) ** 2
        ax.imshow(density[:,:,slice], cmap='hot')
    def plot_density_3D(self, n, iso_val = 0.0002):
        wave_func = np.reshape(self.transition_matrix@self.eigVector[:, n], (self.ninterval, self.ninterval, self.ninterval))
        density = abs(wave_func) ** 2
        verts, faces, _, _ = measure.marching_cubes(density, iso_val)
        fig = plt.figure()
        ax = fig.add_subplot(111, projection='3d')
        ax.plot_trisurf(verts[:, 0], verts[:,1], faces, verts[:, 2],
                        cmap='Spectral', lw=1)

In [ ]:
schro_bs_coulomb = Schrodinger3D_basis_set(coulomb_potential)
schro_bs_coulomb.eig_solve(8)  # 基矢数目为 8^3=512

print("Lowest Energies", schro_bs_coulomb.eigEnergy[:12])

**3d轨道**

**3d orbital**

<!-- bilingual -->

In [ ]:
schro_bs_coulomb.plot_density_3D(9)

## 维度灾难 / Curse of dimensionality

---

数值求解偏微分方程的计算量与问题的维度成指数关系。以n电子波函数为例，假设单一方向上离散点的数目为N，其哈密顿矩阵的尺寸为$N^{3n} \times N^{3n}$。因此，直接对薛定谔方程进行数值求解是没有可操作性的。

The computational cost of numerically solving partial differential equations grows exponentially with the dimensionality of the problem. Taking an n-electron wave function as an example, if the number of discrete points in a single direction is N, the size of its Hamiltonian matrix is $N^{3n} \times N^{3n}$. Therefore, directly solving the Schrödinger equation numerically is not feasible in practice.

解决维度灾难的一个方法是将原始问题降维。还是以电子波函数为例，如果可以将求解多电子波函数简化为求解单电子波函数，则问题就会大大简化。使用平均场方法，我们可以将多电子的运动问题，简化为一个单电子在所有电子构成的有效平均场中的运动问题。这就是第一性模拟计算中广泛使用的Hartree-Fock方法和密度泛函理论的数学基础。

One way to tackle the curse of dimensionality is to reduce the dimensionality of the original problem. Still taking the electron wave function as an example, if solving for a many-electron wave function can be reduced to solving for a single-electron wave function, the problem becomes much simpler. Using a mean-field method, we can reduce the motion of many electrons to the motion of a single electron in an effective mean field generated by all the electrons. This is the mathematical foundation of the Hartree-Fock method and density functional theory (DFT), which are widely used in first-principles simulations.

<!-- bilingual -->